In [ ]:
import colorsys
import pathlib
import tempfile
import urllib.request

In [ ]:
import marimo as mo
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from teeplot import teeplot as tp
from watermark import watermark

In [ ]:
mo.md(
    f"""
```Text
{watermark(
    current_date=True,
    iso8601=True,
    machine=True,
    updated=True,
    python=True,
    iversions=True,
    globals_=globals(),
)}
```
"""
)

```Text
Last updated: 2026-07-25T16:32:24.250789+00:00

Python implementation: CPython
Python version       : 3.10.12
IPython version      : 7.31.1

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.8.0-1062-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

matplotlib: 3.9.2
numpy     : 2.1.2
marimo    : 0.23.2
seaborn   : 0.13.2
teeplot   : 1.4.2
pandas    : 2.2.3

```

# Exploratory sweep and noblip: double-descent analysis

Downloads the collated timeseries parquet files for the
`2026-07-23-exploratory-sweep` and `2026-07-23-exploratory-noblip` SLURM
batch jobs (see `slurm/2026-07-23/`) from OSF, then -- for each swept
condition (blip frequency, L1/L2 regularization mix, zero-init, and
environment schedule mode) -- renders a compound plot summarizing
training/testing dynamics, phenotype composition, and double descent
across model size `v` and training time.

## Fetch data

In [ ]:
# https://osf.io/xn6mk -- 2026-07-23-exploratory-sweep collated timeseries
# https://osf.io/7n63x -- 2026-07-23-exploratory-noblip collated timeseries
OSF_SLUGS = {
    "exploratory-sweep": "xn6mk",
    "exploratory-noblip": "7n63x",
}

In [ ]:
def fetch_osf(slug: str) -> "pathlib.Path":
    """Download a file from OSF by its slug, caching it under the system
    temp directory, and return the path to the cached file.

    Tries a plain stdlib download first; if that fails for any reason
    (auth wall, transient network error, redirect handling, ...) falls
    back to the `osf_fetch` utility from this project's `pylib` (ported
    from the same-named utility added to
    github.com/mmore500/paperproject.git), which is more robust
    (session handling via `requests`, on-disk caching)."""
    cache_path = pathlib.Path(tempfile.gettempdir()) / slug
    url = f"https://osf.io/{slug}/download"
    try:
        if not cache_path.exists():
            print(f"downloading {url} -> {cache_path}")
            request = urllib.request.Request(
                url, headers={"User-Agent": "evodd-bindle/1.0"}
            )
            with urllib.request.urlopen(request, timeout=180) as resp:
                cache_path.write_bytes(resp.read())
        else:
            print(f"reusing cached {cache_path}")
    except Exception as e:
        print(
            f"direct download of {url} failed ({e!r}), falling back to pylib.osf_fetch"
        )
        from pylib import osf_fetch

        cache_path = osf_fetch(slug)
    print(f"size: {cache_path.stat().st_size} bytes")
    return cache_path

In [ ]:
osf_paths = {name: fetch_osf(slug) for name, slug in OSF_SLUGS.items()}

downloading https://osf.io/xn6mk/download -> /tmp/xn6mk
direct download of https://osf.io/xn6mk/download failed (<HTTPError 308: 'Permanent Redirect'>), falling back to pylib.osf_fetch
downloading https://osf.io/xn6mk/download -> /tmp/xn6mk
size: 119502845 bytes
size: 119502845 bytes
downloading https://osf.io/7n63x/download -> /tmp/7n63x
direct download of https://osf.io/7n63x/download failed (<HTTPError 308: 'Permanent Redirect'>), falling back to pylib.osf_fetch
downloading https://osf.io/7n63x/download -> /tmp/7n63x
size: 573252761 bytes
size: 573252761 bytes


## Load data

In [ ]:
df = pd.concat(
    [
        pd.read_parquet(path).assign(dataset=name)
        for name, path in osf_paths.items()
    ],
    ignore_index=True,
)
df["dataset"] = pd.Categorical(df["dataset"])

In [ ]:
pd.concat([df.head(), df.tail()])

,epoch,generation,walltime_sec,pure_train_chi2,test_chi2,blip_train_chi2,test1_frac,test2_frac,test3_frac,test4_frac,...,v,seed,zero_init,l1_scale,l2_scale,blip_freq,num_epoch,schedule_mode,replicate_uid,dataset
0,0,0,61.177933,1.000000,1.000000,1.0,0.0,0.0,0.0,0.0,...,0,1,True,1.000,0.000,0.00,138889,none,0d296efe-b4f7-4e37-ab5d-d9abccd090a2,exploratory-sweep
1,1,138889,61.227436,1.000000,1.000000,1.0,0.0,0.0,0.0,0.0,...,0,1,True,1.000,0.000,0.00,138889,none,0d296efe-b4f7-4e37-ab5d-d9abccd090a2,exploratory-sweep
2,2,277778,61.247986,1.000000,1.000000,1.0,0.0,0.0,0.0,0.0,...,0,1,True,1.000,0.000,0.00,138889,none,0d296efe-b4f7-4e37-ab5d-d9abccd090a2,exploratory-sweep
3,3,416667,61.276153,1.000000,1.000000,1.0,0.0,0.0,0.0,0.0,...,0,1,True,1.000,0.000,0.00,138889,none,0d296efe-b4f7-4e37-ab5d-d9abccd090a2,exploratory-sweep
4,4,555556,61.304268,1.000000,1.000000,1.0,0.0,0.0,0.0,0.0,...,0,1,True,1.000,0.000,0.00,138889,none,0d296efe-b4f7-4e37-ab5d-d9abccd090a2,exploratory-sweep
8844824,2392,499028177,2742.986236,0.998003,0.998008,1.0,0.0,0.0,0.0,0.0,...,0,1,False,0.995,0.005,0.63,138889,global,0eead5ac-c399-45f1-9b92-20e500f05bd6,exploratory-noblip
8844825,2393,499167066,2742.995374,0.998003,0.998008,1.0,0.0,0.0,0.0,0.0,...,0,1,False,0.995,0.005,0.63,138889,global,0eead5ac-c399-45f1-9b92-20e500f05bd6,exploratory-noblip
8844826,2394,499444844,2743.004471,0.998003,0.998008,1.0,0.0,0.0,0.0,0.0,...,0,1,False,0.995,0.005,0.63,138889,global,0eead5ac-c399-45f1-9b92-20e500f05bd6,exploratory-noblip
8844827,2395,499583733,2743.013571,0.998003,0.998008,1.0,0.0,0.0,0.0,0.0,...,0,1,False,0.995,0.005,0.63,138889,global,0eead5ac-c399-45f1-9b92-20e500f05bd6,exploratory-noblip
8844828,2396,500000400,2743.022660,0.998003,0.998008,1.0,0.0,0.0,0.0,0.0,...,0,1,False,0.995,0.005,0.63,138889,global,0eead5ac-c399-45f1-9b92-20e500f05bd6,exploratory-noblip


In [ ]:
df.describe()

,epoch,generation,walltime_sec,pure_train_chi2,test_chi2,blip_train_chi2,test1_frac,test2_frac,test3_frac,test4_frac,...,other_n_classes,l1_loss,l2_loss,regularization_loss,v,seed,l1_scale,l2_scale,blip_freq,num_epoch
count,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,...,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8.844829e+06,8844829.0
mean,1.197987e+03,2.298631e+08,2.917703e+03,4.630608e-01,9.521598e-01,8.952378e-01,1.329452e-01,1.434733e-01,5.178768e-02,1.436339e-01,...,8.617322e+01,6.928008e-02,4.489937e-02,2.062691e-02,1.073181e+01,2.463420e+00,9.961000e-01,3.899987e-03,4.780055e-01,138889.0
std,6.919473e+02,1.484443e+08,2.024705e+03,2.713106e-01,1.766833e+00,1.248017e-01,1.036641e-01,2.391286e-01,4.331800e-02,1.170077e-01,...,2.922392e+02,1.988849e-02,2.999632e-02,7.148784e-03,5.922322e+00,1.128301e+00,2.843720e-03,2.843720e-03,2.449777e-01,0.0
min,0.000000e+00,0.000000e+00,6.106937e+01,2.419075e-02,0.000000e+00,3.266735e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,9.933000e-01,0.000000e+00,0.000000e+00,138889.0
25%,5.990000e+02,9.513896e+07,1.177900e+03,2.900665e-01,6.845400e-02,8.397307e-01,7.750000e-02,4.450000e-02,9.500000e-03,8.150000e-02,...,1.500000e+01,6.843954e-02,2.901163e-02,1.766495e-02,6.000000e+00,1.000000e+00,9.933000e-01,0.000000e+00,5.000000e-01,138889.0
50%,1.198000e+03,2.216668e+08,2.669460e+03,3.815223e-01,3.363440e-01,9.419262e-01,1.250000e-01,9.000000e-02,4.750000e-02,1.305000e-01,...,2.700000e+01,7.322073e-02,3.865855e-02,2.181841e-02,1.000000e+01,2.000000e+00,9.950000e-01,5.000000e-03,6.000000e-01,138889.0
75%,1.797000e+03,3.584725e+08,4.397308e+03,5.739777e-01,7.674860e-01,1.000000e+00,1.690000e-01,1.230000e-01,8.350000e-02,1.780000e-01,...,4.700000e+01,8.024615e-02,5.732106e-02,2.467822e-02,1.600000e+01,3.000000e+00,1.000000e+00,6.700000e-03,6.300000e-01,138889.0
max,2.396000e+03,5.000004e+08,1.438541e+04,1.960300e+00,7.000000e+00,1.925083e+00,8.970000e-01,1.000000e+00,2.355000e-01,7.355000e-01,...,2.000000e+03,1.141377e-01,2.213976e-01,3.813852e-02,2.000000e+01,4.000000e+00,1.000000e+00,6.700000e-03,6.600000e-01,138889.0


## Prepare conditions

A "condition" is one unique combination of the swept, non-model-size
knobs (`blip_freq`, `l1_scale`, `l2_scale`, `zero_init`,
`schedule_mode`) within a dataset -- everything except model size `v`
(the double-descent x-axis) and `seed` (the replicate axis).

In [ ]:
CONDITION_COLS = [
    "dataset",
    "blip_freq",
    "l1_scale",
    "l2_scale",
    "zero_init",
    "schedule_mode",
]

conditions = (
    df[CONDITION_COLS].drop_duplicates().sort_values(CONDITION_COLS)
)

# categorical (qualitative) color map for v, shared across every plot
# below that encodes v via color.
v_values_all = sorted(df["v"].unique())
v_palette = dict(
    zip(
        v_values_all,
        sns.color_palette("tab20", n_colors=len(v_values_all)),
    )
)

In [ ]:
pd.concat([conditions.head(), conditions.tail()])

,dataset,blip_freq,l1_scale,l2_scale,zero_init,schedule_mode
2888284,exploratory-noblip,0.5,0.9933,0.0067,False,global
2885887,exploratory-noblip,0.5,0.9933,0.0067,False,local
2883490,exploratory-noblip,0.5,0.9933,0.0067,False,none
2809183,exploratory-noblip,0.5,0.9933,0.0067,True,global
2806786,exploratory-noblip,0.5,0.9933,0.0067,True,local
160599,exploratory-sweep,0.0,1.0000,0.0000,False,local
158202,exploratory-sweep,0.0,1.0000,0.0000,False,none
4794,exploratory-sweep,0.0,1.0000,0.0000,True,global
2397,exploratory-sweep,0.0,1.0000,0.0000,True,local
0,exploratory-sweep,0.0,1.0000,0.0000,True,none


## Plotting helpers

Phenotype classes `test1_frac`..`test8_frac` are the 8 canonical
`CLASS_8` phenotypes; `train1_frac`..`train3_frac` duplicate
`test1_frac`/`test4_frac`/`test7_frac` by construction (see
`bindle/2026-07-23-exploratory.py`'s `_trace_row`), so the phenotype
stackplot below stacks `test1_frac`..`test8_frac` (not the `train*`
columns, which would double-count) and simply recolors the 3
training-overlap slices distinctly from the other 5 test-only slices.
Blip-pattern matches (`s1_blip_match_frac`..`s3_blip_match_frac`) are a
subset of `other_frac`, so they're broken out of it rather than stacked
on top.

In [ ]:
# CLASS_8 indices 0,3,6 (1-indexed: test1,test4,test7) are the pure
# training patterns S1,S2,S3.
TRAIN_OVERLAP_TEST_IDX = [1, 4, 7]
TEST_ONLY_IDX = [2, 3, 5, 6, 8]

def dull(hex_color, sat_scale=0.35, light_boost=0.25):
    """Desaturate + lighten a bright hex color for the "testing" (as
    opposed to bright "training") stackplot palette."""
    r, g, b = (int(hex_color[i : i + 2], 16) / 255 for i in (1, 3, 5))
    hue, lightness, sat = colorsys.rgb_to_hls(r, g, b)
    lightness = min(1.0, lightness + light_boost)
    sat = sat * sat_scale
    return colorsys.hls_to_rgb(hue, lightness, sat)

BRIGHT_TRAIN_COLORS = ["#e6194B", "#3cb44b", "#4363d8"]
DULL_TEST_COLORS = [dull(c) for c in BRIGHT_TRAIN_COLORS] + [
    dull("#f58231"),
    dull("#911eb4"),
]
GRAY_BLIP_COLORS = ["#404040", "#808080", "#bfbfbf"]
OTHER_COLOR = "#ffffff"

In [ ]:
def make_compound_plot(df_cond, v_palette):
    v_values = sorted(df_cond["v"].unique())
    n_v = len(v_values)

    fig = plt.figure(figsize=(max(10, 2.0 * n_v), 20))
    gs = GridSpec(
        4, 1, figure=fig, height_ratios=[3, 2.4, 2.6, 3.2], hspace=0.55
    )

    # --- row 1: testing (solid) / ACTUAL training (dashed) loss over
    # generations, faceted by v, one line per replicate.
    row1_gs = GridSpecFromSubplotSpec(
        1, n_v, subplot_spec=gs[0], wspace=0.08
    )
    axes1 = [fig.add_subplot(row1_gs[0, 0])]
    for i in range(1, n_v):
        axes1.append(fig.add_subplot(row1_gs[0, i], sharey=axes1[0]))
    seed_values = sorted(df_cond["seed"].unique())
    seed_palette = dict(
        zip(
            seed_values,
            sns.color_palette("tab10", n_colors=len(seed_values)),
        )
    )
    for ax, v in zip(axes1, v_values):
        sub = df_cond[df_cond["v"] == v]
        for seed, g in sub.groupby("seed", observed=True):
            g = g.sort_values("generation")
            color = seed_palette[seed]
            ax.plot(
                g["generation"],
                g["test_chi2"],
                color=color,
                ls="-",
                lw=1.2,
            )
            ax.plot(
                g["generation"],
                g["pure_train_chi2"],
                color=color,
                ls="--",
                lw=1.2,
            )
        ax.set_xscale("symlog")
        ax.set_ylim(bottom=0)
        ax.set_title(f"v={v}", fontsize=9, color=v_palette[v])
        ax.set_xlabel("generation", fontsize=7)
        ax.tick_params(labelsize=7)
        if ax is not axes1[0]:
            plt.setp(ax.get_yticklabels(), visible=False)
    axes1[0].set_ylabel("chi$^2$ error")
    legend_handles = [
        plt.Line2D([], [], color="black", ls="-", label="testing"),
        plt.Line2D(
            [], [], color="black", ls="--", label="training (actual)"
        ),
    ] + [
        plt.Line2D(
            [], [], color=seed_palette[s], label=f"replicate seed={s}"
        )
        for s in seed_values
    ]
    axes1[-1].legend(
        handles=legend_handles,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        fontsize=7,
        frameon=False,
    )

    # --- row 2: median +/- [0th, 100th] percentile band across
    # replicates, training error | testing error, hue=v.
    row2_gs = GridSpecFromSubplotSpec(
        1, 2, subplot_spec=gs[1], wspace=0.15
    )
    ax_train = fig.add_subplot(row2_gs[0, 0])
    ax_test = fig.add_subplot(row2_gs[0, 1])
    for metric_col, ax, label in (
        ("pure_train_chi2", ax_train, "training error (actual)"),
        ("test_chi2", ax_test, "testing error"),
    ):
        agg = (
            df_cond.groupby(["v", "generation"], observed=True)[metric_col]
            .agg(median="median", lo="min", hi="max")
            .reset_index()
        )
        for v, g in agg.groupby("v", observed=True):
            g = g.sort_values("generation")
            color = v_palette[v]
            ax.plot(
                g["generation"],
                g["median"],
                color=color,
                lw=1.5,
                label=f"v={v}",
            )
            ax.fill_between(
                g["generation"],
                g["lo"],
                g["hi"],
                color=color,
                alpha=0.2,
                lw=0,
            )
        ax.set_xscale("symlog")
        ax.set_ylim(bottom=0)
        ax.set_xlabel("generation")
        ax.set_title(label, fontsize=10)
    ax_train.set_ylabel("chi$^2$ error")
    ax_test.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        fontsize=7,
        frameon=False,
        title="v",
    )

    # --- row 3: stackplot of testing phenotype distributions across v,
    # at each replicate's own final recorded generation. Bright =
    # training classes (overlap with test1/test4/test7), dull =
    # test-only classes, grayscale = blip matches, white = remaining
    # "other". Per-(v, seed) last-row (rather than a single shared max
    # generation across the whole condition) since replicates can be
    # truncated at different generations by a SLURM timeout (see
    # bindle/2026-07-23-exploratory.py's progressive save-out).
    ax3 = fig.add_subplot(gs[2])
    final = (
        df_cond.sort_values("generation")
        .groupby(["v", "seed"], observed=True)
        .tail(1)
    )
    frac_cols = [f"test{i}_frac" for i in range(1, 9)]
    blip_cols = [f"s{i}_blip_match_frac" for i in range(1, 4)]
    med = (
        final.groupby("v", observed=True)[
            frac_cols + blip_cols + ["other_frac"]
        ]
        .median()
        .reindex(v_values)
    )
    blip_sum = med[blip_cols].sum(axis=1)
    # blip matches are a subset of "other" (they don't coincide with
    # any of the 8 canonical CLASS_8 phenotypes) -- subtract them back
    # out so the stack doesn't double-count; clip guards against
    # floating-point/rare-coincidence edge cases.
    other_only = (med["other_frac"] - blip_sum).clip(lower=0)

    train_layers = [med[f"test{i}_frac"] for i in TRAIN_OVERLAP_TEST_IDX]
    test_only_layers = [med[f"test{i}_frac"] for i in TEST_ONLY_IDX]
    blip_layers = [med[c] for c in blip_cols]

    stack = train_layers + test_only_layers + blip_layers + [other_only]
    colors = (
        BRIGHT_TRAIN_COLORS
        + DULL_TEST_COLORS
        + GRAY_BLIP_COLORS
        + [OTHER_COLOR]
    )
    labels = (
        [f"train (test{i})" for i in TRAIN_OVERLAP_TEST_IDX]
        + [f"test-only (test{i})" for i in TEST_ONLY_IDX]
        + [f"blip s{i}" for i in range(1, 4)]
        + ["other"]
    )
    polys = ax3.stackplot(v_values, *stack, colors=colors, labels=labels)
    # "other" is white -- outline it so it's visible against the
    # figure background.
    polys[-1].set_edgecolor("black")
    polys[-1].set_linewidth(0.6)
    ax3.set_xlim(min(v_values), max(v_values))
    ax3.set_ylim(0, 1)
    ax3.set_xlabel("v (visible genes)")
    ax3.set_ylabel("phenotype fraction")
    ax3.set_title(
        "final-generation testing phenotype distribution across v",
        fontsize=10,
    )
    ax3.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        fontsize=7,
        frameon=False,
        ncol=1,
    )

    # --- row 4: double descent heatmap -- training time (y) x model
    # size v (x), colored by median testing error. Colormap matches
    # the one used for the double descent heatmaps in Nakkiran et al.
    # 2019 ("Deep Double Descent", arXiv:1912.02292, Figure 2) --
    # matplotlib's viridis (dark purple = low error, yellow = high
    # error).
    ax4 = fig.add_subplot(gs[3])
    grid = (
        df_cond.groupby(["generation", "v"], observed=True)["test_chi2"]
        .median()
        .unstack("v")
        .reindex(columns=v_values)
        .sort_index()
    )
    mesh = ax4.pcolormesh(
        grid.columns.to_numpy(dtype=float),
        grid.index.to_numpy(dtype=float),
        grid.to_numpy(),
        cmap="viridis",
        shading="nearest",
    )
    ax4.set_yscale("symlog")
    ax4.set_xlabel("v (visible genes)")
    ax4.set_ylabel("generation (training time)")
    ax4.set_title("double descent: median testing error", fontsize=10)
    cbar = fig.colorbar(mesh, ax=ax4, pad=0.02)
    cbar.set_label("median test chi$^2$ error")

    return fig

## Compound plots by condition

In [ ]:
for _, _cond in conditions.iterrows():
    _mask = np.logical_and.reduce(
        [df[c] == _cond[c] for c in CONDITION_COLS]
    )
    _df_cond = df[_mask]
    _n_replicates = _df_cond["seed"].nunique()

    # text spacer between compound plots (which are themselves
    # deliberately left untitled) identifying which condition follows.
    mo.output.append(
        mo.md(
            f"""
---
**dataset**=`{_cond['dataset']}`
&nbsp;**blip_freq**=`{_cond['blip_freq']}`
&nbsp;**l1_scale**=`{_cond['l1_scale']}`
&nbsp;**l2_scale**=`{_cond['l2_scale']}`
&nbsp;**zero_init**=`{_cond['zero_init']}`
&nbsp;**schedule_mode**=`{_cond['schedule_mode']}`
&nbsp;**n_replicates(seeds)/v**=`{_n_replicates}`
"""
        )
    )

    with tp.teed(
        make_compound_plot,
        _df_cond,
        v_palette,
        teeplot_outattrs={
            "dataset": str(_cond["dataset"]),
            "blipfreq": str(_cond["blip_freq"]),
            "l1scale": str(_cond["l1_scale"]),
            "l2scale": str(_cond["l2_scale"]),
            "zeroinit": str(_cond["zero_init"]),
            "schedulemode": str(_cond["schedule_mode"]),
        },
        teeplot_subdir=pathlib.Path(__file__).stem,
        teeplot_show=False,
    ) as _fig:
        pass

    # mo.output.append renders the figure into the cell's output
    # immediately, so it's safe (and, across ~dozens of conditions,
    # necessary to avoid unbounded memory growth) to close it right
    # after.
    mo.output.append(_fig)
    plt.close(_fig)

teeplots/2026-07-25-exploratory-sweep-and-noblip/blipfreq=0.5+dataset=exploratory-noblip+l1scale=0.9933+l2scale=0.0067+schedulemode=global+viz=make-compound-plot+zeroinit=False+ext=.pdf
teeplots/2026-07-25-exploratory-sweep-and-noblip/blipfreq=0.5+dataset=exploratory-noblip+l1scale=0.9933+l2scale=0.0067+schedulemode=global+viz=make-compound-plot+zeroinit=False+ext=.png
teeplots/2026-07-25-exploratory-sweep-and-noblip/blipfreq=0.5+dataset=exploratory-noblip+l1scale=0.9933+l2scale=0.0067+schedulemode=local+viz=make-compound-plot+zeroinit=False+ext=.pdf
teeplots/2026-07-25-exploratory-sweep-and-noblip/blipfreq=0.5+dataset=exploratory-noblip+l1scale=0.9933+l2scale=0.0067+schedulemode=local+viz=make-compound-plot+zeroinit=False+ext=.png
teeplots/2026-07-25-exploratory-sweep-and-noblip/blipfreq=0.5+dataset=exploratory-noblip+l1scale=0.9933+l2scale=0.0067+schedulemode=none+viz=make-compound-plot+zeroinit=False+ext=.pdf
teeplots/2026-07-25-exploratory-sweep-and-noblip/blipfreq=0.5+dataset=expl

<marimo-callout-output data-html='"<span class=\"markdown prose dark:prose-invert contents\"><span class=\"paragraph\"><span class=\"text-error\"><strong>Your output is too large</strong></span></span>\n<span class=\"paragraph\">Your output is too large for marimo to show. It has a size\nof 36336808 bytes. Did you output this object by accident?</span>\n<span class=\"paragraph\">If this limitation is a problem for you, you can configure\nthe max output size by adding (eg)</span>\n<div class=\"language-ini codehilite\"><pre><span></span><code><span class=\"k\">[tool.marimo.runtime]</span>\n<span class=\"na\">output_max_bytes</span><span class=\"w\"> </span><span class=\"o\">=</span><span class=\"w\"> </span><span class=\"s\">10_000_000</span>\n</code></pre></div>\n<span class=\"paragraph\">to your pyproject.toml, or with the environment variable\n<code>MARIMO_OUTPUT_MAX_BYTES</code>:</span>\n<div class=\"language-gdscript codehilite\"><pre><span></span><code><span class=\"k\">export</span><span class=\"w\"> </span><span class=\"n\">MARIMO_OUTPUT_MAX_BYTES</span><span class=\"o\">=</span><span class=\"mi\">10</span><span class=\"n\">_000_000</span>\n</code></pre></div>\n<span class=\"paragraph\">Increasing the max output size may cause performance issues.\nIf you run into problems, please reach out\nto us on <a href=\"https://marimo.io/discord?ref=app\" rel=\"noopener noreferrer\" target=\"_blank\">Discord</a> or\n<a href=\"https://github.com/marimo-team/marimo/issues\" rel=\"noopener noreferrer\" target=\"_blank\">GitHub</a>.</span></span>"' data-kind='"warn"'>